# Notebook 03: Retail Simulation - จำลองพฤติกรรมลูกค้าหลายคน

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tiraphap/Gen-Agent-Lectures-at-Econ-TU/blob/master/teaching/notebooks/03_retail_simulation.ipynb)

## สิ่งที่จะได้เรียนรู้
1. **Multi-Agent Simulation**: agents หลายคน ตัดสินใจต่างกันตาม personality
2. **Inventory System**: ของในบ้านหมด → เกิด shopping list อัตโนมัติ
3. **Simulation Loop**: รันหลายรอบ ดู behavior เปลี่ยนตามเวลา
4. **Analysis**: วิเคราะห์ผลลัพธ์ ทำไมลูกค้าแต่ละคนเลือกร้านต่างกัน

---

## Recap จาก Notebook 01-02

- **Notebook 01**: Memory Stream + สูตร retrieval
- **Notebook 02**: Cognitive Loop 5 ขั้นตอน (Perceive → Retrieve → Plan → Execute → Reflect)

ตอนนี้จะ **รวมทุกอย่าง** เข้าด้วยกัน และรัน simulation กับลูกค้าหลายคน!

---
**ผศ.ดร.ถิรภาพ ฟักทอง** | คณะเศรษฐศาสตร์ มหาวิทยาลัยธรรมศาสตร์

In [ ]:
USE_REAL_LLM = False  # ← เปลี่ยนเป็น True ถ้ามี OpenAI API key

if USE_REAL_LLM:
    try:
        import os
        from openai import OpenAI
        client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
        print(f"ใช้ LLM จริง: {MODEL}")
    except:
        USE_REAL_LLM = False

if not USE_REAL_LLM:
    print("ใช้ Mock Responses")

In [ ]:
import json
import random
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import Optional

# ============================================================
# Core Classes (จาก Notebook 01-02)
# ============================================================

@dataclass
class Memory:
    content: str
    created_at: datetime
    importance: int
    memory_type: str = "observation"
    related_store: Optional[str] = None

class MemoryStream:
    RECENCY_DECAY = 0.99
    def __init__(self, name):
        self.name = name
        self.memories: list[Memory] = []
    
    def add(self, content, importance, memory_type="observation", created_at=None, related_store=None):
        mem = Memory(content, created_at or datetime.now(), importance, memory_type, related_store)
        self.memories.append(mem)
        return mem
    
    def retrieve(self, query, top_k=5, current_time=None):
        ct = current_time or datetime.now()
        results = []
        for m in self.memories:
            hrs = (ct - m.created_at).total_seconds() / 3600
            rec = self.RECENCY_DECAY ** hrs
            imp = m.importance / 10.0
            qw = set(query.lower().split())
            mw = set(m.content.lower().split())
            rel = min(1.0, len(qw & mw) / len(qw)) if qw else 0
            results.append((m, rec * imp * rel))
        results.sort(key=lambda x: x[1], reverse=True)
        return results[:top_k]
    
    def format(self, results):
        return "\n".join(f"- {m.content}" for m, _ in results) or "ไม่มี"

print("Core classes loaded!")

In [ ]:
# ============================================================
# Customer Profiles (3 คนที่แตกต่างกัน)
# ============================================================

CUSTOMERS = [
    {
        "name": "สมศรี",
        "age": 45, "job": "แม่บ้าน",
        "household_size": 4,
        "monthly_income": 35000,
        "personality": {
            "price_sensitivity": 0.8,
            "brand_loyalty": 0.6,
            "quality_preference": 0.5,
            "convenience_preference": 0.4,
            "promotion_attraction": 0.9,
            "impulse_buying": 0.3
        },
        "traits": "ใจเย็น ละเอียดรอบคอบ ชอบดูโปร",
        "background": "แม่บ้านดูแลครอบครัว 4 คน ชอบดูใบปลิวโปร ละเอียดเรื่องราคา"
    },
    {
        "name": "สมชาย",
        "age": 32, "job": "โปรแกรมเมอร์",
        "household_size": 1,
        "monthly_income": 80000,
        "personality": {
            "price_sensitivity": 0.3,
            "brand_loyalty": 0.4,
            "quality_preference": 0.8,
            "convenience_preference": 0.9,
            "promotion_attraction": 0.2,
            "impulse_buying": 0.6
        },
        "traits": "ใจร้อน ไม่ชอบเสียเวลา ชอบเทคโนโลยี",
        "background": "โปรแกรมเมอร์อยู่คอนโดคนเดียว ไม่มีเวลา ชอบความสะดวก"
    },
    {
        "name": "ลุงสมบัติ",
        "age": 62, "job": "ข้าราชการเกษียณ",
        "household_size": 2,
        "monthly_income": 15000,
        "personality": {
            "price_sensitivity": 0.95,
            "brand_loyalty": 0.8,
            "quality_preference": 0.4,
            "convenience_preference": 0.3,
            "promotion_attraction": 0.6,
            "impulse_buying": 0.1
        },
        "traits": "ประหยัด มีระเบียบ ยึดติดของเดิม",
        "background": "ข้าราชการเกษียณ อยู่กับภรรยา ประหยัดมาก ภักดีต่อร้านประจำ"
    }
]

STORES = [
    {"name": "CJ More", "type": "supermarket", "price_level": "medium"},
    {"name": "Big C", "type": "hypermarket", "price_level": "low"},
    {"name": "7-Eleven", "type": "convenience", "price_level": "high"},
    {"name": "Makro", "type": "wholesale", "price_level": "very_low"},
]

# แสดง customer profiles
print("╔" + "═" * 70 + "╗")
print("║  Customer Profiles - 3 ลูกค้าที่แตกต่างกัน                         ║")
print("╚" + "═" * 70 + "╝")

for c in CUSTOMERS:
    p = c['personality']
    print(f"\n  {c['name']} ({c['job']}, อายุ {c['age']})")
    print(f"    รายได้: {c['monthly_income']:,} บาท | ครอบครัว: {c['household_size']} คน")
    print(f"    นิสัย: {c['traits']}")
    print(f"    Personality:")
    for trait, val in p.items():
        bar = "█" * int(val * 10) + "░" * (10 - int(val * 10))
        print(f"      {trait:<25} {bar} {val:.1f}")

In [ ]:
# ============================================================
# Inventory System - ระบบของใช้ในบ้าน
# ============================================================
#
# ทุกวัน ของในบ้านจะถูกใช้ไป (consumption)
# ถ้าเหลือน้อยกว่า threshold → ใส่ shopping list อัตโนมัติ

@dataclass
class InventoryItem:
    """ของในบ้าน 1 รายการ"""
    name: str
    current: float        # จำนวนที่เหลือ
    threshold: float      # ถ้าต่ำกว่านี้ → ต้องซื้อ
    consumption_per_day: float  # ใช้ต่อวันต่อคน
    unit: str


class Inventory:
    """
    ระบบจัดการของใช้ในบ้าน
    
    ทุกวัน:
    1. คำนวณ consumption (ใช้ไปเท่าไหร่)
    2. หัก current stock
    3. ถ้า current < threshold → เพิ่มเข้า shopping list
    """
    
    def __init__(self, household_size: int):
        self.household_size = household_size
        self.items: dict[str, InventoryItem] = {}
        self.shopping_list: list[dict] = []
    
    def add_item(self, name, current, threshold, consumption_per_day, unit):
        self.items[name] = InventoryItem(name, current, threshold, consumption_per_day, unit)
    
    def consume_daily(self):
        """
        จำลองการใช้ของในบ้าน 1 วัน
        
        สูตร: consumed = consumption_per_day × household_size × random(0.8, 1.2)
        (คูณ random เพื่อให้มีความหลากหลาย)
        """
        self.shopping_list = []  # reset
        
        for name, item in self.items.items():
            # คำนวณการใช้ (มี randomness เล็กน้อย)
            consumed = item.consumption_per_day * self.household_size * random.uniform(0.8, 1.2)
            item.current = max(0, item.current - consumed)
            
            # ถ้าเหลือน้อยกว่า threshold → ต้องซื้อ!
            if item.current <= item.threshold:
                urgency = "high" if item.current == 0 else "medium"
                self.shopping_list.append({
                    "item": name,
                    "urgency": urgency,
                    "current": item.current,
                    "unit": item.unit
                })
    
    def restock(self, item_name, quantity):
        """เติมของหลังซื้อ"""
        if item_name in self.items:
            self.items[item_name].current += quantity
    
    def status(self):
        """แสดงสถานะปัจจุบัน"""
        lines = []
        for name, item in self.items.items():
            pct = min(1.0, item.current / (item.threshold * 3))  # normalize
            bar = "█" * int(pct * 10) + "░" * (10 - int(pct * 10))
            warn = " ⚠️ ต้องซื้อ!" if item.current <= item.threshold else ""
            lines.append(f"    {name:<15} {bar} {item.current:.1f}/{item.threshold:.1f} {item.unit}{warn}")
        return "\n".join(lines)


# ทดลอง: สร้าง inventory ให้สมศรี (ครอบครัว 4 คน)
inv = Inventory(household_size=4)
inv.add_item("นม", current=3.0, threshold=1.0, consumption_per_day=0.25, unit="กล่อง")
inv.add_item("ไข่", current=8.0, threshold=4.0, consumption_per_day=0.5, unit="ฟอง")
inv.add_item("ข้าวสาร", current=5.0, threshold=2.0, consumption_per_day=0.15, unit="กก.")
inv.add_item("ผงซักฟอก", current=1.5, threshold=0.5, consumption_per_day=0.05, unit="กก.")

print("Inventory ของ สมศรี (วันแรก):")
print(inv.status())

# จำลอง 5 วัน
print("\n" + "─" * 50)
print("จำลองการใช้ของ 5 วัน:")
for day in range(1, 6):
    inv.consume_daily()
    print(f"\n  วันที่ {day}:")
    print(inv.status())
    if inv.shopping_list:
        print(f"    → Shopping list: {[s['item'] for s in inv.shopping_list]}")

In [ ]:
# ============================================================
# Mock Responses สำหรับ 3 agents
# ============================================================
#
# แต่ละ agent ตัดสินใจต่างกันตาม personality!

MOCK_RESPONSES = {
    "สมศรี": {
        "thinking_steps": [
            "นมเหลือน้อย ต้องซื้อเพิ่มแน่ๆ",
            "เห็นว่า Big C ลดนม 30% น่าสนใจมาก (promotion_attraction สูง)",
            "ยอมไปไกลหน่อยเพื่อของถูก (price_sensitivity สูง)",
            "ตัดสินใจไป Big C"
        ],
        "decision": {"action": "go_shopping", "store": "Big C",
                     "items_to_buy": ["นม", "ไข่", "ผงซักฟอก"], "estimated_spend": 850}
    },
    "สมชาย": {
        "thinking_steps": [
            "นมหมดแล้ว ต้องซื้อ",
            "ไม่อยากเสียเวลาไปไกล (convenience สูงมาก)",
            "7-Eleven ใต้คอนโด เดิน 1 นาทีถึง",
            "แพงกว่านิดหน่อยก็ไม่เป็นไร (price_sensitivity ต่ำ)"
        ],
        "decision": {"action": "go_shopping", "store": "7-Eleven",
                     "items_to_buy": ["นม", "ขนมปัง"], "estimated_spend": 120}
    },
    "ลุงสมบัติ": {
        "thinking_steps": [
            "ข้าวสารเหลือน้อย ต้องซื้อ",
            "ไปร้านประจำ CJ More เหมือนเคย (brand_loyalty สูง)",
            "ไม่อยากเปลี่ยนร้าน รู้จักพนักงานดีแล้ว",
            "ซื้อแต่ของที่จำเป็นจริงๆ (impulse_buying ต่ำมาก)"
        ],
        "decision": {"action": "go_shopping", "store": "CJ More",
                     "items_to_buy": ["ข้าวสาร", "ไข่"], "estimated_spend": 350}
    }
}

print("Mock responses for 3 agents loaded!")

In [ ]:
# ============================================================
# รัน Simulation: 3 agents, 1 รอบ
# ============================================================

print("╔" + "═" * 70 + "╗")
print("║  RETAIL SIMULATION - 3 Agents, Same Situation              ║")
print("║  สถานการณ์เดียวกัน แต่ตัดสินใจต่างกัน!                       ║")
print("╚" + "═" * 70 + "╝")

results = []

for customer in CUSTOMERS:
    name = customer['name']
    mock = MOCK_RESPONSES[name]
    
    print(f"\n{'─'*70}")
    print(f"  Agent: {name} ({customer['job']})")
    print(f"{'─'*70}")
    
    # Step 1: Perceive
    print(f"  [PERCEIVE] สังเกตสิ่งรอบตัว...")
    
    # Step 2: Retrieve
    print(f"  [RETRIEVE] ดึงความทรงจำที่เกี่ยวข้อง...")
    
    # Step 3: Plan (LLM)
    print(f"  [PLAN] LLM กำลังคิด...")
    for i, step in enumerate(mock['thinking_steps'], 1):
        print(f"    💭 {i}. {step}")
    
    d = mock['decision']
    print(f"  [EXECUTE] → ไป {d['store']} ซื้อ {', '.join(d['items_to_buy'])}")
    print(f"             จ่าย {d['estimated_spend']:,} บาท")
    
    results.append({
        "name": name,
        "store": d['store'],
        "spend": d['estimated_spend'],
        "items": d['items_to_buy']
    })

# ============================================================
# เปรียบเทียบผลลัพธ์
# ============================================================
print(f"\n{'═'*70}")
print(f"  COMPARISON: ทำไมตัดสินใจต่างกัน?")
print(f"{'═'*70}")

print(f"\n  {'Agent':<15} {'Store':<15} {'Spend':<10} {'Items'}")
print(f"  {'─'*65}")
for r in results:
    print(f"  {r['name']:<15} {r['store']:<15} {r['spend']:<10,} {', '.join(r['items'])}")

print(f"\n  สังเกต:")
print(f"  - สมศรี → Big C: เพราะ promotion_attraction=0.9 (ชอบโปร) + price_sensitivity=0.8 (ดูราคา)")
print(f"  - สมชาย → 7-Eleven: เพราะ convenience=0.9 (สะดวกสำคัญ) + price_sensitivity=0.3 (ไม่แคร์ราคา)")
print(f"  - ลุงสมบัติ → CJ More: เพราะ brand_loyalty=0.8 (ภักดีร้านประจำ) + impulse_buying=0.1 (ซื้อเฉพาะที่จำเป็น)")

In [ ]:
# ============================================================
# Visualization: Personality Comparison
# ============================================================

print("╔" + "═" * 70 + "╗")
print("║  Personality Comparison (Text Radar Chart)                  ║")
print("╚" + "═" * 70 + "╝")

traits = ["price_sensitivity", "brand_loyalty", "quality_preference",
          "convenience_preference", "promotion_attraction", "impulse_buying"]

trait_labels = {
    "price_sensitivity": "ดูราคา",
    "brand_loyalty": "ภักดีแบรนด์",
    "quality_preference": "ชอบของดี",
    "convenience_preference": "สะดวกสำคัญ",
    "promotion_attraction": "ชอบโปร",
    "impulse_buying": "ซื้อตามใจ"
}

print(f"\n  {'Trait':<20}", end="")
for c in CUSTOMERS:
    print(f"  {c['name']:<20}", end="")
print()
print("  " + "─" * 80)

for trait in traits:
    label = trait_labels[trait]
    print(f"  {label:<20}", end="")
    for c in CUSTOMERS:
        val = c['personality'][trait]
        bar = "█" * int(val * 10) + "░" * (10 - int(val * 10))
        print(f"  {bar} {val:.1f}     ", end="")
    print()

## สรุป: Multi-Agent Retail Simulation

### Key Takeaways

1. **Personality Traits กำหนด Behavior**:
   - คนรักโปร → ไป hypermarket
   - คนรักสะดวก → ไป convenience store
   - คนภักดีร้าน → ไปร้านประจำ

2. **Memory ส่งผลต่อการตัดสินใจ**: ประสบการณ์ดี/แย่ที่ร้าน → จำ → ส่งผลครั้งหน้า

3. **Inventory Consumption** → สร้าง Shopping List อัตโนมัติ → Trigger การไปซื้อของ

4. **LLM ทำให้ realistic**: ไม่มี if-else ตายตัว แต่ละครั้ง LLM อาจตัดสินใจต่างกัน

### Applications
- **Marketing**: ทดสอบโปรโมชั่น ก่อนลองจริง
- **Store Planning**: เข้าใจว่าลูกค้าแต่ละกลุ่มเลือกร้านอย่างไร
- **Product Placement**: สินค้าอะไร ควรวางที่ร้านไหน
- **Policy Testing**: เปลี่ยนราคา/โปร → ดูว่าพฤติกรรมเปลี่ยนไหม